# Đánh giá toàn bộ quy trình: sinh ứng viên và chọn lại câu

Chạy đúng quy trình suy luận của mô hình đề xuất trên một tập Karpathy (test hoặc val) và đánh giá ở các điểm
được báo cáo trong bài:

| Điểm | Cách chọn câu |
|---|---|
| `before_rerank` | ứng viên đầu tiên (beam search thường), chưa chọn lại |
| `three_score` | ba tín hiệu kế thừa: 0,2·S_base + 0,4·S_align + 0,4·S_belief |
| `plus_s_retr` | ba tín hiệu + 0,1·S_retr |
| `plus_s_obj` | ba tín hiệu + 1·S_obj (ngưỡng r = 3) |
| `five_score` | **năm tín hiệu**, S_obj với r = 3 (cấu hình chính của bài) |
| `five_score_1of5` | năm tín hiệu, S_obj với r = 1 (để so sánh ngưỡng) |

Các bước cho mỗi ảnh:

1. **Bằng chứng** E(I) = YOLO-World (ngưỡng 0,25) ∪ vật thể được ≥ r trong K = 5 câu truy hồi nhắc tới. Câu truy hồi
   lấy từ kho câu mô tả của tập huấn luyện theo cosine CLIP ViT-B/32.
2. **Luật chặn cố định** khi giải mã: trừ δ vào logits của từ vật thể COCO không thuộc E(I). `RUN_BOTH = True`
   chạy đủ quy trình hai lần, chưa chặn (δ = 0) rồi có chặn (δ = `DELTA`), mỗi δ lưu một file. `RUN_BOTH = False` chỉ
   chạy `DELTA` (đặt `DELTA = 0` để chỉ chạy chưa chặn). Chạy cả hai tốn gấp đôi thời gian. `EVIDENCE_SOURCES`
   chọn nguồn bằng chứng cho luật chặn và S_obj ở δ = `DELTA`: `"union"` (mô hình đề xuất), `"yolo"` (chỉ YOLO-World,
   C8), `"retr"` (chỉ câu truy hồi, C9). Mỗi nguồn là một lượt sinh câu và một file riêng (`..._d3_yolo_test.json`).
3. **Sinh 5 ứng viên**: 1 câu beam search thường (beam 3), cộng diverse beam search (10 beam, 5 nhóm, phạt đa dạng
   0,8).
4. **Ba tín hiệu kế thừa** (bộ chọn lại câu dạng công thức): S_base là log-xác suất trung bình, S_align là CLIPScore
   (CLIP ViT-B/16), S_belief là belief revision theo vật thể YOLO.
5. **Hai tín hiệu đề xuất:**
   - S_obj = −(số vật thể câu nhắc tới nhưng không thuộc E(I)), với E(I) ở đây lấy YOLO-World ngưỡng `SOBJ_CONF` = 0,55
     (chọn trên val). Luật chặn và S_belief vẫn dùng ngưỡng 0,25.
   - S_retr = cosine trung bình giữa câu và K câu truy hồi.
6. **Tổng hợp:** mỗi tín hiệu chuẩn hóa min-max trong năm ứng viên của ảnh, cộng có trọng số, chọn câu có điểm
   cao nhất.

**Cấu hình** (cell CẤU HÌNH): `CHECKPOINT_NAME`, `SPLIT`, `DELTA`; trọng số `W_BASE/W_ALIGN/W_BELIEF`,
`W_RETR`, `W_OBJ`; `SOBJ_CONF`; `COMPUTE_SPICE` (cần Java 8, chậm). Mặc định là luật chặn δ = 3 +
chọn lại câu, đúng cấu hình của bảng khóa luận. `CHECKPOINT_NAME` là checkpoint epoch 40 của một seed (41, 42, 43):
`seed<N>_dynlogits_contrastive_attnpool_epoch40_model.pth` (mô hình đề xuất) hoặc `baseline_epoch40_seed<N>.pth`
(mô hình cơ sở CE).

**Độ đo:** BLEU-1..4, METEOR, ROUGE-L, CIDEr (tách từ kiểu `simple`), SPICE tùy chọn, CHAIRs, CHAIRi. Nhãn đúng
của CHAIR là vật thể trong COCO instances hợp với vật thể nhắc tới trong 5 câu tham chiếu.

**Lưu kết quả:** file JSON có `metrics` cho từng điểm và `rows` theo định dạng report của repo (ứng viên, xếp hạng,
vật thể YOLO). Vì vậy các script trong `research/scripts/` dùng lại được file này. Vòng sinh câu lưu giữa chừng và
chạy tiếp được nếu bị ngắt.

**Môi trường:** notebook gọi thẳng các module gốc của repo (không chép lại), nên số liệu khớp với
`retrieval_logit_bias_experiment.py`, `logit_bias_final_eval.py`, `evgate/signal_table.py` và
`sobj_threshold_eval.py`. Cần `torch`, `transformers`, `ultralytics`, `pycocoevalcap`, `nltk`, Java cho METEOR.

**Chạy trên Kaggle** (GPU T4 hoặc P100, bật Internet). Gắn các dataset:
- COCO 2014 (ảnh `val2014`, `instances_val2014.json`) và Karpathy split (`dataset_coco.json`), như notebook huấn luyện;
- một dataset code + trọng số tự tải lên, gồm các module gốc của repo (`local_captioner.py`, `generate.py`,
  `clipscore.py`, `formula_reranker.py`, `chair_metric.py`, `chair_synonyms.txt`, `yolo_objects.py`, `vocab.py`,
  `schema.py`, `evaluate_reranked_quality.py`, để chung một thư mục), file từ điển
  và `yolov8l-worldv2.pt`;
- checkpoint cần đánh giá, gắn riêng (dataset hoặc Kaggle Model) để dễ thay, tên file khớp `CHECKPOINT_NAME`. Có thể thêm `train_captions.json` + `train_caption_emb_b32.npy` để khỏi mã hóa lại kho truy hồi.

Notebook tự tìm file theo tên trong `/kaggle/input`, kết quả ghi vào `/kaggle/working/results/`. Chạy thử với
`MAX_IMAGES = 5` trước.

Notebook này được sinh bởi `research/scripts/build_eval_notebooks.py`; sửa ở script đó rồi chạy lại để giữ hai
notebook đồng bộ.


In [ ]:
# ==========================================
# CẤU HÌNH
# ==========================================
import json, os, shutil, sys, time
from collections import Counter
from pathlib import Path

# Mô hình cần đánh giá. Mô hình cơ sở CE: dynlogits_contrastive_attnpool_epoch40_model.pth; mô hình đề xuất DL+CL:
# dynlogits_contrastive_attnpool_epoch40_model.pth. Cả hai dùng chung từ điển dưới đây.
# Ghi đường dẫn đầy đủ trong /kaggle/input hoặc chỉ tên file (notebook tự tìm trong mọi input đã gắn).
CHECKPOINT_NAME = "/kaggle/input/models/nganhthnh/seed41-model/pytorch/default/1/seed41_dynlogits_contrastive_attnpool_epoch40_model.pth"
VOCAB_NAME = "/kaggle/input/datasets/nganhthnh/mehc-cap-eval/coco2014_karpathy_restval_dynlogits_contrastive_attnpool_vocab.json"
YOLO_NAME = "/kaggle/input/datasets/nganhthnh/mehc-cap-eval/yolov8l-worldv2.pt"   # YOLO-World

# Notebook tìm file theo tên trong mọi dataset đã gắn ở /kaggle/input, nên không phụ thuộc tên dataset.
FILES = {}
for root, dirs, files in os.walk("/kaggle/input"):
    dirs[:] = [d for d in dirs if d not in ("train2014", "test2014")]   # bỏ qua ảnh không dùng cho nhanh
    for f in files:
        FILES.setdefault(f, Path(root) / f)


def find(name):
    """Đường dẫn đầy đủ thì dùng luôn, không thì tìm theo tên file."""
    p = Path(name)
    if p.is_file():
        return p
    assert p.name in FILES, f"Không thấy {p.name} trong /kaggle/input, hãy gắn dataset/model chứa nó"
    return FILES[p.name]


CODE_DIR = find("local_captioner.py").parent        # dataset code: các module .py gốc của repo
KARPATHY_JSON = find("dataset_coco.json")
INSTANCES_JSON = find("instances_val2014.json")
# ảnh val/test Karpathy đều thuộc val2014; lấy thư mục chứa nhiều ảnh nhất để không vớ nhầm ảnh mẫu lẻ
IMAGE_DIR = Counter(p.parent for n, p in FILES.items() if n.startswith("COCO_val2014_")).most_common(1)[0][0]
# Checkpoint gắn riêng (dataset hoặc Kaggle Model), đổi mô hình thì gắn cái mới rồi sửa CHECKPOINT_NAME
CHECKPOINT, VOCAB_FILE, YOLO_MODEL = find(CHECKPOINT_NAME), find(VOCAB_NAME), find(YOLO_NAME)
# Kho truy hồi đã mã hóa (train_captions.json + train_caption_emb_b32.npy): gắn sẵn thì dùng, không thì tự mã hóa
# (chỉ nhận thư mục có đủ cả hai file, vì có dataset chỉ chứa bản sao file .npy)
RETRIEVAL_CACHE = next((p.parent for p in [FILES.get("train_captions.json")] if p and (p.parent / "train_caption_emb_b32.npy").is_file()),
                       Path("/kaggle/working/retrieval_cache"))
RESULTS_DIR = Path("/kaggle/working/results")

SPLIT = "test"        # "test" | "val"
START, MAX_IMAGES = 0, None   # ảnh xếp theo tên file như các report của bài; vd MAX_IMAGES = 5 để chạy thử
DELTA = 3.0           # mức chặn của luật cố định; 0 = không chặn

YOLO_CONF, YOLO_MAX_OBJECTS = 0.25, 20   # ngưỡng cho luật chặn và S_belief
K_RETR = 5            # số câu truy hồi
MAJ = 3               # ngưỡng đa số r: vật thể được >= r câu truy hồi nhắc tới thì vào tập bằng chứng
MAX_NEW_TOKENS = 40
FORCE_CPU = False     # True: không dùng GPU

# Bộ chọn lại câu (giá trị đã khóa trên tập kiểm định, mục 4.2 của bài)
W_BASE, W_ALIGN, W_BELIEF = 0.2, 0.4, 0.4
W_RETR, W_OBJ = 0.1, 1.0
SOBJ_CONF = 0.55          # ngưỡng YOLO-World riêng cho tập bằng chứng của S_obj (chọn trên val); luật chặn và S_belief vẫn dùng YOLO_CONF
CLIP_SCORER = "openai/clip-vit-base-patch16"
COMPUTE_SPICE = False      # True: thêm SPICE cho before_rerank, three_score, five_score (Java 8, khoảng 1 giờ/5.000 ảnh)
SAVE_EVERY = 100           # lưu giữa chừng sau mỗi chừng này ảnh; chạy lại cell sinh câu sẽ đi tiếp từ chỗ dừng
RUN_BOTH = True            # True: chạy đủ quy trình hai lần, chưa chặn (δ = 0) rồi có chặn (δ = DELTA); gấp đôi thời gian
# Nguồn bằng chứng cho luật chặn và S_obj ở δ = DELTA: "union" (YOLO-World ∪ truy hồi, mô hình đề xuất), "yolo" (chỉ
# YOLO-World, C8), "retr" (chỉ câu truy hồi, C9). Mỗi nguồn là một lượt sinh câu riêng; δ = 0 không chặn nên chỉ chạy "union".
EVIDENCE_SOURCES = ["union"]
RUNS = list(dict.fromkeys(([(0.0, "union")] if RUN_BOTH else []) + [(DELTA, src) for src in EVIDENCE_SOURCES]))


def output_of(run):
    delta, src = run
    return RESULTS_DIR / f"eval_rerank_{CHECKPOINT.stem}_d{delta:g}{'' if src == 'union' else '_' + src}_{SPLIT}.json"


def label(run):
    return f"δ = {run[0]:g}" + ("" if run[1] == "union" else f", bằng chứng {run[1]}")

In [ ]:
# ==========================================
# NẠP CODE REPO, DỮ LIỆU VÀ NHÃN CHAIR
# ==========================================
import subprocess   # cần bật Internet trong Settings của notebook
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "pycocoevalcap", "python-dotenv"], check=True)
if shutil.which("java") is None:   # METEOR cần Java
    subprocess.run("apt-get -qq update && apt-get -qq install -y default-jre-headless > /dev/null", shell=True)
import nltk
for pkg in ("punkt", "punkt_tab", "wordnet", "omw-1.4"):
    nltk.download(pkg, quiet=True)
USE_METEOR = shutil.which("java") is not None
if not USE_METEOR:
    print("[!] Không có Java: bỏ METEOR")
if FORCE_CPU:
    import torch
    torch.cuda.is_available = lambda: False   # các module của repo tự chọn thiết bị bằng hàm này
sys.path.insert(0, str(CODE_DIR))

import numpy as np
import torch
from tqdm.auto import tqdm

import chair_metric
import local_captioner
import vocab as vocab_mod
from evaluate_reranked_quality import compute_metrics
from PIL import Image
from yolo_objects import YoloObjectExtractor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
data = json.loads(KARPATHY_JSON.read_text(encoding="utf-8"))
items = sorted(({"image": im["filename"], "references": [s["raw"] for s in im["sentences"]]}
                for im in data["images"] if im["split"] == SPLIT), key=lambda r: r["image"])
items = items[START: START + MAX_IMAGES] if MAX_IMAGES else items[START:]
paths = [IMAGE_DIR / r["image"] for r in items]
missing = [p.name for p in paths if not p.is_file()]
assert not missing, f"Thiếu {len(missing)} ảnh, vd {missing[:3]}"

OBJECTS, INV = chair_metric.load_synonyms()
fn2id = chair_metric.load_filename_to_cocoid(KARPATHY_JSON)
inst = chair_metric.load_instance_objects_by_cocoid(INSTANCES_JSON, INV)
# Nhãn đúng của CHAIR = vật thể trong COCO instances  U  vật thể nhắc tới trong 5 câu tham chiếu
GT = {r["image"]: chair_metric.ground_truth_from_captions(r["references"], OBJECTS, INV) | inst.get(fn2id.get(r["image"]), set())
      for r in items}


class Clip:
    """CLIP ViT-B/32 cho kho truy hồi và S_retr: vector ảnh/câu đã chuẩn hóa L2, lưu float16."""
    def __init__(self, name="openai/clip-vit-base-patch32"):
        from transformers import CLIPModel, CLIPProcessor
        self.device = DEVICE
        self.model = CLIPModel.from_pretrained(name).to(self.device).eval()
        self.proc = CLIPProcessor.from_pretrained(name)

    @torch.no_grad()
    def _encode(self, fn, batches):
        out = []
        for inp in batches:
            with torch.autocast(self.device, enabled=self.device == "cuda"):
                f = fn(**inp.to(self.device))
            f = f if isinstance(f, torch.Tensor) else f.pooler_output   # transformers >= 5 trả về object, pooler_output là vector đã chiếu (như clipscore.py)
            out.append(torch.nn.functional.normalize(f.float(), dim=-1).cpu().numpy().astype(np.float16))
        return np.concatenate(out)

    def text(self, texts, bs=512):
        return self._encode(self.model.get_text_features, (self.proc(text=texts[i: i + bs], return_tensors="pt", padding=True,
                                                                     truncation=True) for i in range(0, len(texts), bs)))

    def images(self, paths, bs=64):
        return self._encode(self.model.get_image_features, (self.proc(images=[Image.open(p).convert("RGB") for p in paths[i: i + bs]],
                                                                      return_tensors="pt") for i in range(0, len(paths), bs)))


def minmax(v):
    """Chuẩn hóa min-max trong các ứng viên của một ảnh; bằng nhau hết thì trả 0,5."""
    lo, hi = v.min(), v.max()
    return np.full_like(v, 0.5) if hi - lo < 1e-9 else (v - lo) / (hi - lo)


def objs_of(text):
    """Danh sách vật thể COCO (đã quy về tên lớp) mà một câu nhắc tới, như CHAIR."""
    return chair_metric.caption_to_words(text, OBJECTS, INV)[1]


def score(captions, meteor=USE_METEOR, spice=False):
    """BLEU-1..4, METEOR, ROUGE-L, CIDEr (×100), SPICE tùy chọn, CHAIRs/CHAIRi; tách từ kiểu 'simple' như bài."""
    refs = [r["references"] for r in items]
    m = compute_metrics(captions, refs, use_meteor=meteor, use_cider=True, use_spice=spice, tokenizer="simple")
    ch = chair_metric.compute_chair({r["image"]: c for r, c in zip(items, captions)}, GT, OBJECTS, INV)
    out = {k: round(float(v) * (100 if k == "CIDEr" else 1), 2) for k, v in m.items()}
    out.update(CHAIRs=round(ch["CHAIRs"], 2), CHAIRi=round(ch["CHAIRi"], 2),
               mean_len=round(float(np.mean([len(c.split()) for c in captions])), 2))
    return out


print(f"{len(items)} ảnh {SPLIT} | thiết bị {DEVICE} | checkpoint {CHECKPOINT.name} | δ = {DELTA:g}")

In [ ]:
# ==========================================
# BẰNG CHỨNG VẬT THỂ E(I) = YOLO-World  U  {vật thể được >= MAJ trong K câu truy hồi nhắc tới}
# ==========================================
# Kho truy hồi: toàn bộ câu mô tả của tập huấn luyện (train + restval), mã hóa bằng CLIP ViT-B/32, lưu cache.
clip32 = Clip()
caps_f, emb_f = RETRIEVAL_CACHE / "train_captions.json", RETRIEVAL_CACHE / "train_caption_emb_b32.npy"
if caps_f.is_file() and emb_f.is_file():
    BANK_CAPS, BANK_EMB = json.loads(caps_f.read_text(encoding="utf-8")), np.load(emb_f)
else:
    BANK_CAPS = [s["raw"].strip().lower() for im in data["images"] if im["split"] in ("train", "restval") for s in im["sentences"]]
    print(f"Mã hóa kho truy hồi: {len(BANK_CAPS)} câu (chỉ làm một lần)")
    BANK_EMB = clip32.text(BANK_CAPS)
    RETRIEVAL_CACHE.mkdir(parents=True, exist_ok=True)
    caps_f.write_text(json.dumps(BANK_CAPS), encoding="utf-8")
    np.save(emb_f, BANK_EMB)

# K câu gần ảnh nhất theo cosine ảnh-câu (CLIP B/32); kho không chứa câu của tập val/test nên không rò nhãn.
img_emb = clip32.images(paths)
dt = torch.float16 if clip32.device == "cuda" else torch.float32
bank_t = torch.from_numpy(BANK_EMB).to(clip32.device, dt)
NB = np.concatenate([(torch.from_numpy(img_emb[i: i + 256]).to(clip32.device, dt) @ bank_t.T).topk(K_RETR, dim=1).indices.cpu().numpy()
                     for i in range(0, len(img_emb), 256)])
del bank_t

extractor = YoloObjectExtractor(str(YOLO_MODEL), YOLO_CONF, YOLO_MAX_OBJECTS)
vocab_mod.set_object_noun_vocab(extractor.class_names())   # danh từ vật thể cho bằng chứng âm của S_belief
YOLO_OBJS = [extractor.extract(p) for p in tqdm(paths, desc="YOLO-World")]
YOLO_SET = [{INV[o.label] for o in objs if o.confidence >= YOLO_CONF and o.label in INV} for objs in YOLO_OBJS]
NB_COUNTS = [Counter(o for j in idx for o in set(objs_of(BANK_CAPS[j]))) for idx in NB]   # số câu truy hồi nhắc tới mỗi vật thể
RETR_SET = [{o for o, n in c.items() if n >= MAJ} for c in NB_COUNTS]   # vật thể được >= MAJ câu truy hồi nhắc tới
EVIDENCE = [y | r for y, r in zip(YOLO_SET, RETR_SET)]
EVIDENCE_OF = {"union": EVIDENCE, "yolo": YOLO_SET, "retr": RETR_SET}   # nguồn bằng chứng: Y+R, chỉ Y (C8), chỉ R (C9)
print(f"Trung bình {np.mean([len(e) for e in EVIDENCE]):.2f} vật thể trong E(I) mỗi ảnh")

In [ ]:
# ==========================================
# LUẬT CHẶN CỐ ĐỊNH TRÊN LOGITS:  z'(w) = z(w) - δ · 1[w là từ vật thể COCO, o(w) không thuộc E(I)]
# ==========================================
# Áp ở mọi bước giải mã của cả beam search thường và diverse beam search; từ không phải vật thể giữ nguyên.
IRREGULAR = {"men": "man", "women": "woman", "kids": "kid", "guys": "guy", "mice": "mouse"}
VOCAB = json.loads(VOCAB_FILE.read_text(encoding="utf-8"))
TOK2OBJ = {}
for word, idx in VOCAB.items():
    w = IRREGULAR.get(word) or chair_metric._LEMMATIZER.lemmatize(word, pos="n")
    if w in OBJECTS:
        TOK2OBJ[idx] = INV[w]
print(f"{len(TOK2OBJ)} token trong từ điển ứng với vật thể COCO")

CURRENT_BIAS = None
if not hasattr(local_captioner, "_ORIG_DECODE_STEP"):
    local_captioner._ORIG_DECODE_STEP = local_captioner.ImageCaptioningTransformer.decode_step


def _biased_decode_step(self, memory, sequence):
    logits = local_captioner._ORIG_DECODE_STEP(self, memory, sequence)
    return logits if CURRENT_BIAS is None else logits + CURRENT_BIAS


local_captioner.ImageCaptioningTransformer.decode_step = _biased_decode_step


def rule_bias(evidence, delta=None):
    delta = DELTA if delta is None else delta
    if not delta:
        return None
    bias = torch.zeros(len(VOCAB))
    for tid, obj in TOK2OBJ.items():
        if obj not in evidence:
            bias[tid] = -delta
    return bias.to(DEVICE)

In [ ]:
# ==========================================
# SINH 5 ỨNG VIÊN (có luật chặn) VÀ CHẤM BA TÍN HIỆU
# ==========================================
from clipscore import ClipScorer
from formula_reranker import FormulaReranker
from generate import CandidateCaptionGenerator

generator = CandidateCaptionGenerator(checkpoint_path=CHECKPOINT, vocab_path=VOCAB_FILE, max_candidates=5, num_beams=10,
                                      num_beam_groups=5, diversity_penalty=0.8, max_new_tokens=MAX_NEW_TOKENS, safe_beam_size=3)
clip16 = ClipScorer(model_name=CLIP_SCORER)
formula = FormulaReranker(alpha=W_BASE, beta=W_ALIGN, gamma=W_BELIEF, formula_mode="linear", clip_scorer=clip16,
                          belief_neg_confidence=0.0)

RUN_INFO = {"sobj_conf": SOBJ_CONF, "w_retr": W_RETR, "w_obj": W_OBJ}   # chỉ đổi bước chọn câu, không cần sinh lại


def run_key(run):
    key = {"checkpoint": str(CHECKPOINT), "split": SPLIT, "start": START, "images": len(items), "delta": run[0],
           "maj": MAJ, "k_retr": K_RETR, "yolo_model": YOLO_MODEL.name, "weights": [W_BASE, W_ALIGN, W_BELIEF]}
    if run[1] != "union":   # file Y+R giữ khóa cũ để chạy tiếp được các file đã có
        key["evidence"] = run[1]
    return key


def save(run, rows, metrics=None):
    out = output_of(run)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps({"run": run_key(run), "rerank": RUN_INFO, "metrics": metrics, "rows": rows}, ensure_ascii=False),
                   encoding="utf-8")


def generate_rows(run):
    """Sinh 5 ứng viên với luật chặn (δ, nguồn bằng chứng) của run và chấm ba tín hiệu; lưu giữa chừng, chạy tiếp được."""
    global CURRENT_BIAS   # _biased_decode_step đọc biến toàn cục này
    delta, src = run
    rows, out = [], output_of(run)
    if out.is_file():
        old = json.loads(out.read_text(encoding="utf-8"))
        if old.get("run") == run_key(run):
            rows = old["rows"]
            print(f"{label(run)}: chạy tiếp từ ảnh {len(rows)}")
    t0, done0 = time.time(), len(rows)
    for i in tqdm(range(done0, len(items)), desc=f"Sinh và chấm ứng viên, {label(run)}"):
        r, path, objs = items[i], paths[i], YOLO_OBJS[i]
        CURRENT_BIAS = rule_bias(EVIDENCE_OF[src][i], delta)
        cands = generator.generate(path, objs)
        CURRENT_BIAS = None
        scored = clip16.score(path, cands)
        ranked = formula.rerank(scored, objs)
        rows.append({
            "image": r["image"], "references": r["references"],
            "baseline_candidate": cands[0].text, "caption": ranked[0].caption.text,
            "objects": [{"label": o.label, "confidence": o.confidence, "box_xyxy": list(o.box_xyxy) if o.box_xyxy else None} for o in objs],
            "candidates": [c.__dict__ for c in scored],
            "ranking": [{"caption": x.caption.text, "rank": x.rank, "rerank_score": x.rerank_score, "reason": x.reason} for x in ranked],
            "neighbours": [int(j) for j in NB[i]], "evidence": sorted(EVIDENCE_OF[src][i]),
        })
        if (i + 1) % SAVE_EVERY == 0:
            save(run, rows)
            sec = (time.time() - t0) / (i + 1 - done0)   # tqdm không hiện trong log khi Save & Run All, nên in thêm dòng này
            print(f"{label(run)}: {i + 1}/{len(items)} ảnh | {sec:.2f} giây/ảnh | còn khoảng {sec * (len(items) - i - 1) / 60:.0f} phút",
                  flush=True)
    CURRENT_BIAS = None
    save(run, rows)
    if len(rows) > done0:
        print(f"{label(run)}: {(time.time() - t0) / (len(rows) - done0):.2f} giây mỗi ảnh")
    return rows


ROWS_BY_RUN = {run: generate_rows(run) for run in RUNS}

In [ ]:
# ==========================================
# HAI TÍN HIỆU ĐỀ XUẤT VÀ CÁC CÁCH CHỌN CÂU
# ==========================================
# S_retr(c) = cosine trung bình giữa câu c và K câu truy hồi (CLIP ViT-B/32)
# S_obj(c)  = -(số vật thể COCO c nhắc tới nhưng không thuộc E(I)); E(I) dùng ngưỡng r (MAJ hoặc 1)
texts = sorted({c["text"] for rows in ROWS_BY_RUN.values() for row in rows for c in row["candidates"]})
TEXT_EMB = dict(zip(texts, clip32.text(texts)))
YOLO_SET_SOBJ = [{INV[o.label] for o in objs if o.confidence >= SOBJ_CONF and o.label in INV} for objs in YOLO_OBJS]


def selections(rows, src):
    per_image = []
    for row, idx, ysup, cnt in zip(rows, NB, YOLO_SET_SOBJ, NB_COUNTS):
        cands = [c["text"] for c in row["candidates"]]
        rs = {x["caption"]: x["rerank_score"] for x in row["ranking"]}
        base = np.array([rs[t] for t in cands], dtype=np.float64)          # ba tín hiệu (đã chuẩn hóa trong ảnh)
        ce = np.stack([TEXT_EMB[t] for t in cands]).astype(np.float32)
        s_retr = (ce @ BANK_EMB[idx].astype(np.float32).T).mean(axis=1)
        s_obj = {}
        for thr in sorted({MAJ, 1}):
            ret = {o for o, n in cnt.items() if n >= thr}
            sup = {"union": ysup | ret, "yolo": ysup, "retr": ret}[src]
            s_obj[thr] = np.array([-sum(o not in sup for o in objs_of(t)) for t in cands], dtype=np.float64)
        per_image.append({"cands": cands, "base": base, "s_retr": s_retr, "s_obj": s_obj})

    def pick(fn):
        return [p["cands"][int(np.argmax(fn(p)))] for p in per_image]

    sel = {
        "before_rerank": [row["baseline_candidate"] for row in rows],
        "three_score": pick(lambda p: p["base"]),
        "plus_s_retr": pick(lambda p: p["base"] + W_RETR * minmax(p["s_retr"])),
        "plus_s_obj": pick(lambda p: p["base"] + W_OBJ * minmax(p["s_obj"][MAJ])),
        "five_score": pick(lambda p: p["base"] + W_RETR * minmax(p["s_retr"]) + W_OBJ * minmax(p["s_obj"][MAJ])),
        "five_score_1of5": pick(lambda p: p["base"] + W_RETR * minmax(p["s_retr"]) + W_OBJ * minmax(p["s_obj"][1])),
    }
    equal = round(100 * float(np.mean([p["s_obj"][MAJ].max() == p["s_obj"][MAJ].min() for p in per_image])), 1)
    return sel, equal


SELECTIONS = {run: selections(rows, run[1]) for run, rows in ROWS_BY_RUN.items()}
for run, (_, equal) in SELECTIONS.items():
    print(f"{label(run)}: S_obj bằng nhau ở cả năm ứng viên (không đổi thứ hạng) ở {equal}% số ảnh")

In [ ]:
# ==========================================
# TÍNH ĐIỂM VÀ LƯU (mỗi lượt một file)
# ==========================================
SPICE_ON = {"before_rerank", "three_score", "five_score"}
KEYS = ("BLEU-4", "METEOR", "ROUGE-L", "CIDEr", "CHAIRs", "CHAIRi")
METRICS = {}
for run, (sel, equal) in SELECTIONS.items():
    print(f"===== {label(run)}" + (" (chưa chặn)" if not run[0] else " (có chặn)"))
    METRICS[run] = {}
    for name, caps in sel.items():
        METRICS[run][name] = score(caps, spice=COMPUTE_SPICE and name in SPICE_ON)
        print(f"{name:16s}", {k: METRICS[run][name].get(k) for k in KEYS})
    METRICS[run]["sobj_equal_pct"] = equal
    rows = ROWS_BY_RUN[run]
    for row, cap in zip(rows, sel["five_score"]):
        row["caption_five_score"] = cap
    save(run, rows, METRICS[run])
    print(f"Đã lưu {output_of(run)}")